# Ollama Model Debug

Прямые запросы к моделям через Ollama OpenAI API.  
Видно сырой вывод: какие поля в delta, какие теги в content, как устроен reasoning.

In [1]:
import json
import os
import time
import urllib.request
from IPython.display import display, HTML, Markdown

os.environ["no_proxy"] = "localhost,127.0.0.1"
os.environ["NO_PROXY"] = "localhost,127.0.0.1"

OLLAMA_URL = "http://localhost:11434/v1/chat/completions"

In [3]:
# List available models
req = urllib.request.Request("http://localhost:11434/v1/models")
with urllib.request.urlopen(req, timeout=5) as resp:
    models_data = json.loads(resp.read())

available = [m["id"] for m in models_data["data"]]
print("Available models:")
for m in available:
    print(f"  • {m}")

Available models:
  • gpt-oss:20b
  • gemma4:31b
  • qwen3.6:27b-coding-nvfp4
  • qwen2.5-coder:7b


## Settings

Выбери модель, системное сообщение и промпт.

In [38]:
MODEL = "qwen3.6:27b-coding-nvfp4"  # <-- change to any model from the list above

SYSTEM = (
    "You a senior dev coder. You think extensevily before answering. You answer in Russian"
)

USER_PROMPT = "Сколько будет 15 * 7? Подумай пошагово, потом дай ответ."

print(f"Model:  {MODEL}")
print(f"System: {SYSTEM[:80]}...")
print(f"Prompt: {USER_PROMPT}")

Model:  qwen3.6:27b-coding-nvfp4
System: You a senior dev coder. You think extensevily before answering. You answer in Ru...
Prompt: Сколько будет 15 * 7? Подумай пошагово, потом дай ответ.


## Stream request & collect raw chunks

In [39]:
payload = json.dumps({
    "model": MODEL,
    "stream": True,
    "messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": USER_PROMPT},
    ],
}).encode()

req = urllib.request.Request(
    OLLAMA_URL,
    data=payload,
    headers={"Content-Type": "application/json"},
)

chunks = []
full_content = ""
full_reasoning = ""
all_delta_keys = set()

t0 = time.time()
first_token_time = None
lines = []
with urllib.request.urlopen(req, timeout=60) as resp:
    for raw_line in resp:
        line = raw_line.decode("utf-8", errors="replace").strip()
        lines.append(line)
        if not line or line == "data: [DONE]":
            continue
        if line.startswith("data: "):
            line = line[6:]
        try:
            chunk = json.loads(line)
        except json.JSONDecodeError:
            continue

        chunks.append(chunk)
        choice = (chunk.get("choices") or [{}])[0]
        delta = choice.get("delta", {})
        all_delta_keys.update(delta.keys())

        content = delta.get("content", "")
        reasoning = delta.get("reasoning_content", "") or delta.get("reasoning", "")

        if (content or reasoning) and first_token_time is None:
            first_token_time = time.time() - t0

        full_content += content
        full_reasoning += reasoning

elapsed = time.time() - t0

print(f"Done in {elapsed:.1f}s | First token: {first_token_time:.2f}s | Chunks: {len(chunks)}")
print(f"Delta keys seen: {all_delta_keys}")

Done in 65.6s | First token: 8.06s | Chunks: 731
Delta keys seen: {'role', 'reasoning', 'content'}


In [40]:
print(full_reasoning)

Here's a thinking process:

1.  **Analyze User Input:**
   - User asks: "Сколько будет 15 * 7?" (What is 15 * 7?)
   - User requests: "Подумай пошагово, потом дай ответ." (Think step by step, then give the answer.)
   - User language: Russian
   - My role: Senior dev coder, think extensively, answer in Russian.

2.  **Identify Core Task:**
   - Calculate 15 multiplied by 7.
   - Provide a step-by-step reasoning process.
   - Output in Russian.

3.  **Step-by-Step Calculation (Mental):**
   - 15 * 7 can be broken down:
     - Method 1: Distributive property: (10 + 5) * 7 = 10*7 + 5*7 = 70 + 35 = 105
     - Method 2: Standard multiplication: 7 * 5 = 35 (write 5, carry 3), 7 * 1 = 7, plus carried 3 = 10. So 105.
     - Method 3: 15 * 7 = 7 * 15 = 105 (known fact)
   - I'll choose Method 1 for clear step-by-step explanation as it's pedagogical and transparent.

4.  **Draft Response (Internal Refinement in Russian):**
   - Начну с пошагового разбора:
     - Умножение 15 на 7 можно разложить

In [42]:
print(full_content)

Проведу пошаговый расчёт, как обычно делаю при отладке или разборе алгоритмов:

1. **Разложим множитель по разрядам**:  
   \(15 = 10 + 5\)

2. **Применим распределительное свойство умножения**:  
   \(15 \times 7 = (10 + 5) \times 7 = 10 \times 7 + 5 \times 7\)

3. **Вычислим каждую часть отдельно**:  
   - \(10 \times 7 = 70\)  
   - \(5 \times 7 = 35\)

4. **Сложим промежуточные результаты**:  
   \(70 + 35 = 105\)

5. **Перекрёстная проверка (умножение в столбик)**:  
   - \(7 \times 5 = 35\) → записываем `5`, запоминаем перенос `3`  
   - \(7 \times 1 = 7\) → прибавляем перенос `3` → получаем `10`  
   - Итог: `105`

✅ **Ответ: 105**


## Analysis

In [ ]:
has_reasoning_field = bool(full_reasoning)
has_think = "<think>" in full_content
has_thinking = "<thinking>" in full_content
has_reason = "<reason>" in full_content or "<reasoning>" in full_content

tags_found = []
if has_reasoning_field: tags_found.append("`reasoning` field in delta")
if has_think:           tags_found.append("`<think>` tag in content")
if has_thinking:        tags_found.append("`<thinking>` tag in content")
if has_reason:          tags_found.append("`<reason>`/`<reasoning>` tag in content")
if not tags_found:      tags_found.append("❌ No reasoning separation detected")

print(f"Model: {MODEL}")
print(f"Reasoning methods: {', '.join(tags_found)}")
print(f"Content length: {len(full_content)} chars")
print(f"Reasoning length: {len(full_reasoning)} chars")

## Raw `content` field (full)

In [ ]:
print(full_content)

## Raw `reasoning` / `reasoning_content` field (full)

In [ ]:
if full_reasoning:
    print(full_reasoning)
else:
    print("(empty — this model does not use the reasoning field)")

## Content with tags highlighted

Теги подсвечены цветом для наглядности.

In [ ]:
import html as _html

def highlight_tags(text: str) -> str:
    """Escape HTML, then colorize known reasoning tags."""
    escaped = _html.escape(text)
    for tag in ["think", "thinking", "reason", "reasoning"]:
        escaped = escaped.replace(
            f"&lt;{tag}&gt;",
            f'<span style="background:#ff6b6b;color:white;padding:1px 4px;border-radius:3px">&lt;{tag}&gt;</span>',
        )
        escaped = escaped.replace(
            f"&lt;/{tag}&gt;",
            f'<span style="background:#ff6b6b;color:white;padding:1px 4px;border-radius:3px">&lt;/{tag}&gt;</span>',
        )
    return f'<pre style="white-space:pre-wrap;font-size:13px;line-height:1.5">{escaped}</pre>'

display(HTML(highlight_tags(full_content)))

## First 5 raw chunks (JSON)

Чтобы увидеть структуру delta объекта.

In [ ]:
for i, chunk in enumerate(chunks[:5]):
    print(f"--- chunk {i} ---")
    print(json.dumps(chunk, indent=2, ensure_ascii=False))
    print()

## Save results to JSON

In [ ]:
out_name = MODEL.replace(":", "_").replace("/", "_")
out_path = f"{out_name}.json"

with open(out_path, "w") as f:
    json.dump({
        "model": MODEL,
        "system": SYSTEM,
        "user_prompt": USER_PROMPT,
        "elapsed_s": round(elapsed, 2),
        "first_token_s": round(first_token_time, 2) if first_token_time else None,
        "chunk_count": len(chunks),
        "all_delta_keys": sorted(all_delta_keys),
        "has_reasoning_field": has_reasoning_field,
        "has_think_tag": has_think,
        "has_thinking_tag": has_thinking,
        "has_reason_tag": has_reason,
        "full_content": full_content,
        "full_reasoning": full_reasoning,
        "chunks": chunks,
    }, f, ensure_ascii=False, indent=2)

print(f"Saved to {out_path}")